In [6]:
import requests
import pathlib
import xml.etree.ElementTree as ET
import json
import time
import string
import sys
import pandas as pd
from deutsche_bahn_api import ApiAuthentication
from pathlib import Path
import os
import numpy as np
import re
from datetime import datetime, timedelta
from math import radians, cos, sin, asin, sqrt


BASE_DB_API = "https://apis.deutschebahn.com/db-api-marketplace/apis/"
STOP_PLACES_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stop-places"
TIMETABLES_V1_URL = BASE_DB_API + "timetables/v1"
RIS_STATION_URL = BASE_DB_API + "ris-station/v1/stations/"
ROOT_DIR = Path(os.getcwd())
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = ROOT_DIR / "Raw_Data"

DB_CLIENT_ID = "a9f83c55d26c3ee7f48f4ce887ec2a57"
DB_API_KEY = "422cac21a0a83876c75efb8806589ea0"
PKP_API_KEY = "KMjLifnR-a6RGlVgs36-OGG82nZ2gZRuiySF_y-tWSbUMX4LNS3JfBk1hli1B59YIXfdIrIy3ZvwUpkrMueAeA"

header = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/xml"
}

header2 = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/json"
}

def get_all_stations(country_code="DE"):
    params = {"state": country_code}
    
    response = requests.get(RIS_STATION_URL, headers=header, params=params)
    if response.status_code == 200:
        return response.json().get('stations', [])
    else:   
        print(f"Error: {response.status_code}")
        return []

def haversine(lat1, lon1, lat2, lon2):
    """ Calculate distance in km between two points """
    R = 6371 # Earth radius
    dLat = radians(lat2 - lat1)
    dLon = radians(lon2 - lon1)
    a = sin(dLat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dLon/2)**2
    return 2 * R * asin(sqrt(a))

# api credentials validation
api_authentication = ApiAuthentication( DB_CLIENT_ID, DB_API_KEY)
success:bool = api_authentication.test_credentials()
success

True

#  Railway stations selection & filtering

First, match train station with selected cities from simplemaps ( Processed_Data/cities_500K). Later for matched cities retrieve required timetables.

Matching should be done geospatially

In [7]:
PROCESSED_DATA_DIR = ROOT_DIR / "Processed_Data"

cities_data = pd.read_csv(PROCESSED_DATA_DIR / "cities_500K.csv")
cities_data.rename(columns={"name_city":"name", "lat_city":"latitude", "lng_city":"longitude","country":"iso_code"}, inplace=True)
cities_data.head()

,name,latitude,longitude,iso_code,population,is_capital
0,Vienna,48.2083,16.3725,AT,1973403.0,True
1,Brussels,50.8467,4.3525,BE,1235192.0,True
2,Antwerp,51.2178,4.4003,BE,536079.0,False
3,Sofia,42.7000,23.3300,BG,1383435.0,True
4,Prague,50.0875,14.4214,CZ,1357326.0,True


In [8]:
RIS_STATION_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stations"
header_ris = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/vnd.de.db.ris+json"
}

res = requests.get(url=RIS_STATION_URL, headers=header_ris, params={"countryCode":"AT"})
res.text

'{"offset":0,"limit":100,"total":5703,"stations":[{"stationID":"1","names":{"DE":{"name":"Aachen Hbf"}},"metropolis":{},"address":{"street":"Bahnhofstr.","houseNumber":"2a","postalCode":"52064","city":"Aachen","state":"Nordrhein-Westfalen","country":"DE"},"stationCategory":"CATEGORY_2","availableTransports":[],"availableLocalServices":[],"transportAssociations":[],"owner":{"name":"DB InfraGO Personenbahnhöfe","organisationalUnit":{"id":4,"name":"RB West","nameShort":"RB West"}},"countryCode":"DE","state":"NW","timeZone":"Europe/Berlin","position":{"longitude":6.091499,"latitude":50.7678},"validFrom":"2018-12-31T23:00:00Z","mobilityServiceStaffOnSite":true},{"stationID":"1000","names":{"DE":{"name":"Burkhardswalde-Maxen"}},"metropolis":{},"address":{"street":"Gesundbrunnen","houseNumber":"60c","postalCode":"01809","city":"Müglitztal-Burkhardswalde","state":"Sachsen","country":"DE"},"stationCategory":"CATEGORY_7","availableTransports":[],"availableLocalServices":[],"transportAssociations

Testing stop-places accesspoint

In [10]:
response = requests.get(url=f"{STOP_PLACES_URL}/by-name/Warsaw?sortBy=RELEVANCE&onlyActive=true&withSynonyms=true&limit=3", headers=header_ris)
response.text

'{"stopPlaces":[{"evaNumber":"5100065","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Centralna","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":21.003234,"latitude":52.22886}},{"evaNumber":"5100067","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Zachodnia","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":20.965247,"latitude":52.219972}},{"evaNumber":"5100066","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Wschodnia","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":21.052335,"latitude":52.251548}}]}'

Using RIS:stations retrieve all cities main stations

In [17]:
def get_stations_api_stop_places(cities : pd.DataFrame, 
                      base_url: str = STOP_PLACES_URL,  
                      limit: int = 3) -> pd.DataFrame:
    """
    Retrieves station information from the Deutsche Bahn API based on the station name.
    
    Keyword arguments:
    cities -- DataFrame containing city information
    base_url -- Base URL for the API
    limit -- Maximum number of stations to retrieve per city
    Return: DataFrame of stations for given cities
    """
    retrieved_stations = pd.DataFrame(columns=['city_name', 'eva_id', 'station_name', 'latitude', 'longitude'])
    for _, row in cities.iterrows():
        city_name = row['name']
        response = None
        params = {
            "sortBy": "RELEVANCE",
            "onlyActive": "true",
            "withSynonyms": "true",
            "latitude": row['latitude'],
            "longitude": row['longitude'],
            "limit": limit
        }
        success = False
        retries = 0

        while not success and retries < 3:
            response = requests.get(url=f"{base_url}/by-name/{city_name}", params=params, headers=header_ris)

            if response.status_code == 429:
                print(f"Rate limit reached. Sleeping for 1s...")
                time.sleep(1)
                retries += 1
                continue

            if response.status_code == 200:
                stations = response.json().get('stopPlaces', [])
                print(f"City: {city_name}, Stations Found: {len(stations)}")
                for station in stations:
                    latitude = float(station.get('position').get('latitude'))
                    longitude = float(station.get('position').get('longitude'))

                    if haversine(row['latitude'], row['longitude'], latitude, longitude) > 10:
                        print(f"Skipping station {station.get('names').get('DE').get('nameLong')} due to distance.")
                        continue

                    retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({
                        'city_name': [city_name],
                        'eva_id': [str(station.get('evaNumber'))],
                        'station_name': [str(station.get('names').get('DE').get('nameLong'))],
                        'latitude': [float(station.get('position').get('latitude'))],
                        'longitude': [float(station.get('position').get('longitude'))]
                    })])
                success = True
            else:
                print(f"Error retrieving stations for city {city_name}: {response.status_code}")
                break

            time.sleep(0.09)  # To respect API rate limits

    return retrieved_stations

In [12]:
test_cities = cities_data.iloc[np.random.choice(cities_data.shape[0], 5, replace=False)]
test_cities

,name,latitude,longitude,iso_code,population,is_capital
14,Essen,51.4508,7.0131,DE,584580.0,False
55,Bucharest,44.4325,26.1039,RO,1716961.0,True
54,Lisbon,38.7253,-9.1500,PT,548703.0,True
31,Marseille,43.2964,5.3700,FR,873076.0,False
19,Duisburg,51.4347,6.7625,DE,502211.0,False


In [96]:
output  = get_stations_api_stop_places(test_cities, limit=3)

City: Málaga, Stations Found: 0
City: Essen, Stations Found: 3


C:\Users\Adam\AppData\Local\Temp\ipykernel_15984\3773918198.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({


City: Warsaw, Stations Found: 3
City: Paris, Stations Found: 3
City: Vilnius, Stations Found: 1
Skipping station Vilniuser Straße, Erfurt due to distance.


For cities above 500K we select at least 3 station if possible as timetables for those stations might contain most crutial connections

In [18]:
citiies_stations = get_stations_api_stop_places(cities_data, limit=1)

City: Vienna, Stations Found: 1


C:\Users\Adam\AppData\Local\Temp\ipykernel_3492\580966334.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({


City: Brussels, Stations Found: 1
City: Antwerp, Stations Found: 1
City: Sofia, Stations Found: 1
Skipping station Sofie-Hammer-Straße, Osnabrück due to distance.
City: Prague, Stations Found: 1
City: Berlin, Stations Found: 1
City: Stuttgart, Stations Found: 1
City: Munich, Stations Found: 1
City: Hamburg, Stations Found: 1
City: Cologne, Stations Found: 1
City: Frankfurt, Stations Found: 1
City: Düsseldorf, Stations Found: 1
City: Leipzig, Stations Found: 1
City: Dortmund, Stations Found: 1
City: Essen, Stations Found: 1
City: Bremen, Stations Found: 1
City: Dresden, Stations Found: 1
City: Hannover, Stations Found: 1
City: Nuremberg, Stations Found: 1
City: Duisburg, Stations Found: 1
City: Copenhagen, Stations Found: 1
City: Tallinn, Stations Found: 1
Skipping station Tallinner Straße, Schwerin (Meckl) due to distance.
City: Madrid, Stations Found: 1
Skipping station Madrider Ring, Würzburg due to distance.
City: Barcelona, Stations Found: 1
City: Valencia, Stations Found: 1
Skippi

In [19]:
citiies_stations

,city_name,eva_id,station_name,latitude,longitude
0,Vienna,8103000,Wien Hbf,48.185101,16.377113
0,Brussels,8800004,Bruxelles Midi,50.835376,4.335694
0,Antwerp,8800007,Antwerpen Centraal,51.215811,4.421168
0,Prague,5400014,Praha hl.n.,50.083062,14.436039
0,Berlin,8011160,Berlin Hbf,52.525592,13.369545
0,Stuttgart,8000096,Stuttgart Hbf,48.784780,9.182757
0,Munich,8000261,München Hbf,48.140232,11.558335
0,Hamburg,8002549,Hamburg Hbf,53.552736,10.006909
0,Cologne,8003368,Köln Messe/Deutz,50.940874,6.975001
0,Frankfurt,8000105,Frankfurt(Main)Hbf,50.106682,8.662828


In [130]:
citiies_stations[citiies_stations['city_name'] == "Prague" ]

,city_name,eva_id,station_name,latitude,longitude
0,Prague,5400014,Praha hl.n.,50.083062,14.436039


Found Train Stations for given cities

first cities which had no stations at DB API

In [148]:
cities_no_station = pd.merge(cities_data, citiies_stations, how='outer',left_on=['name'], right_on=['city_name'], indicator=True).query('_merge == "left_only"')
cities_no_station.reset_index(inplace=True)
cities_no_station

,index,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y,_merge
0,6,Athens,37.9842,23.7281,GR,643452.0,True,NaN,NaN,NaN,NaN,NaN,left_only
1,24,Bucharest,44.4325,26.1039,RO,1716961.0,True,NaN,NaN,NaN,NaN,NaN,left_only
2,49,Gothenburg,57.7075,11.9675,SE,607882.0,False,NaN,NaN,NaN,NaN,NaN,left_only
3,56,Helsinki,60.1708,24.9375,FI,664921.0,True,NaN,NaN,NaN,NaN,NaN,left_only
4,62,Lisbon,38.7253,-9.1500,PT,548703.0,True,NaN,NaN,NaN,NaN,NaN,left_only
5,71,Madrid,40.4169,-3.7033,ES,3266126.0,True,NaN,NaN,NaN,NaN,NaN,left_only
6,79,Málaga,36.7194,-4.4200,ES,586384.0,False,NaN,NaN,NaN,NaN,NaN,left_only
7,80,Naples,40.8333,14.2500,IT,913462.0,False,NaN,NaN,NaN,NaN,NaN,left_only
8,89,Riga,56.9489,24.1064,LV,660187.0,True,NaN,NaN,NaN,NaN,NaN,left_only
9,94,Sevilla,37.3900,-5.9900,ES,684025.0,False,NaN,NaN,NaN,NaN,NaN,left_only


In [146]:
cities_with_stations = pd.merge(cities_data, citiies_stations, how='inner',left_on=['name'], right_on=['city_name'])
cities_with_stations

,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y
0,Vienna,48.2083,16.3725,AT,1973403.0,True,Vienna,8103000,Wien Hbf,48.185101,16.377113
1,Vienna,48.2083,16.3725,AT,1973403.0,True,Vienna,8101818,Wien St. Marx,48.187984,16.399550
2,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800004,Bruxelles Midi,50.835376,4.335694
3,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800002,Bruxelles-Nord,50.860239,4.361454
4,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800003,Bruxelles-Central,50.845492,4.357064
...,...,...,...,...,...,...,...,...,...,...,...
102,Ljubljana,46.0514,14.5061,SI,284293.0,True,Ljubljana,7900141,Ljubljana Litostroj,46.077196,14.489906
103,Ljubljana,46.0514,14.5061,SI,284293.0,True,Ljubljana,7900266,Ljubljana Stegne,46.086817,14.479965
104,Bratislava,48.1439,17.1097,SK,475503.0,True,Bratislava,5600207,Bratislava hl.st.,48.158910,17.106466
105,Bratislava,48.1439,17.1097,SK,475503.0,True,Bratislava,5600582,Bratislava-Petrzalka,48.120679,17.100448


# Rertieve Timetables for existing city stations

currently timetables for DB offers limited data. Other timetables are required to be used

In [122]:
header_timetables = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/xml"
}


def fetch_hourly_plan(eva_no, date_str, hour_str):
    """
    Fetches the planned timetable for a specific station, date (YYMMDD), and hour (HH).
    """
    url = f"{TIMETABLES_V1_URL}/plan/{eva_no}/{date_str}/{hour_str}"
    success = False
    rertries = 0
    plan_hour = ""

    
    while not success and rertries < 3:
        response = requests.get(url, headers=header_timetables)
        status_code  = response.status_code

        if status_code == 200:
            success = True
            retries = 0
            plan_hour =  response.text
        elif status_code == 404 or status_code == 401:
            print("Access Error: Check API")
            break 
        elif status_code == 429:
            time.sleep(1)
        retries += 1
            
    return plan_hour

def parse_hour_plan(hour_xml:str):
    pass

# test run 
fetch_hourly_plan("8103000","251229","12")


'<?xml version=\'1.0\' encoding=\'UTF-8\'?><timetable station=\'Wien Hbf\'><s id="-641886370829105986-2512291130-3"><tl f="F" t="p" o="81" c="EC" n="204"/><ar pt="2512291158" pp="12A-B" fb="EC 204" ppth="Wien Westbahnhof|Wien Meidling"/><dp pt="2512291210" pp="12A-B" fb="EC 204" pde="Krakow Glowny" ppth="Breclav|Hodonin|Stare Mesto u Uherského Hradiste|Otrokovice|Prerov|Hranice na Morave|Ostrava-Svinov|Ostrava hl.n.|Bohumin"/></s><s id="-721807130937592434-2512291213-1"><tl f="F" t="p" o="81" c="ICE" n="90"/><dp pt="2512291213" pp="8A-B" fb="ICE 90" ppth="Wien Meidling|St.Pölten Hbf|Linz Hbf|Passau Hbf|Plattling|Regensburg Hbf|Nürnberg Hbf|Coburg|Erfurt Hbf|Leipzig Hbf|Lutherstadt Wittenberg Hbf|Berlin Südkreuz|Berlin Hbf|Berlin Gesundbrunnen"/></s><s id="-6130228582536441801-2512290618-10"><tl f="F" t="p" o="51" c="EC" n="203"/><ar pt="2512291149" pp="6A-B" fb="EC 203" pde="Krakow Glowny" ppth="Bohumin|Ostrava hl.n.|Ostrava-Svinov|Hranice na Morave|Prerov|Otrokovice|Stare Mesto u Uher